# Parameter Tuning Harness
<!-- structured-notebook -->
## Notebook Summary
Purpose: sweep BERTopic hyperparameters (UMAP + HDBSCAN + vectorizer) and save each round's outputs for comparison.

Main steps:
- Reuse embeddings produced by the main pipeline (avoids recomputing SentenceTransformer embeddings on 72k docs).
- Iterate over parameter combinations, fitting a new BERTopic per configuration.
- Save each round under `round_{N}/` with model, topics.csv, probabilities, and manifest.

This harness produced rounds 1–11 of the Reddit topic-modelling exploration. Round 11 is the production model; earlier rounds are retained for comparison.


# Round progression
<!-- structured-notebook -->
## Summary

| Round | What changed | Outcome |
|---|---|---|
| 1 | Baseline, no proper preprocessing or HP tuning | Exploratory only |
| 2 | Proper preprocessing + UMAP + HDBSCAN | First usable |
| 3 | HP tuning attempt | Failed on vectorizer params |
| 4 | Like round 2, 40k max features, saving artifacts for downstream analysis; no soft probs | Too few topics, some dominant + vague |
| 5 | Parameter tuning + bugfixes | Intermediate |
| 6–8 | Further tuning to reduce outliers, push toward finer topics | Improving |
| 9 | More tuning toward finer topics | Good noise control |
| 10 | 78 topics, 30% noise, `eom` selection | Production candidate |
| **11** | `cluster_selection_method="leaf"`, soft probabilities, `min_dist=0.0` in UMAP | **Production** |

The `min_dist=0.0` change in UMAP made the embedding space more clusterable — this was the fix for a late-stage hiccup where topics were failing to separate cleanly. Round 11 is the only round using `cluster_selection_method="leaf"`; all earlier rounds used `"eom"`. Leaf produces finer-grained clusters.


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
import hashlib
import os
import json
import random
from sentence_transformers import SentenceTransformer
import hdbscan
import pandas as pd
import umap
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
import re
from langdetect import DetectorFactory
import pickle
import numpy as np

DetectorFactory.seed = 0
random.seed(0)
np.random.seed(0)

current_round = 11
current_round_path = f"round_{current_round}"
os.makedirs(current_round_path, exist_ok=True)
raw_path = REDDIT_OUTPUT / "merged_submissions.jsonl"


# Load mappings/labels
train_docs = pickle.load(open("round_4/unique_docs.pkl","rb"))
X = np.load("round_4/embeddings_fp32_l2.npy", mmap_mode="r")   # (U, D), float32
map_o2u = np.load("round_4/map_orig_to_unique.npy")
kept_idx = np.load(f"round_4/kept_indices.npy")          # (K,)
ts_K = np.load(f"round_4/kept_ts_seconds.npy")       # (K,)
assert len(train_docs) == X.shape[0] == len(train_docs)
assert kept_idx.shape[0] == map_o2u.shape[0] == ts_K.shape[0]
n_docs = len(train_docs)


# X must match unique_docs exactly
assert X.shape[0] == len(train_docs), "Embeddings/text count mismatch"
# Optional: confirm normalization stayed intact (~1.0)
print("mean L2 norm:", float(np.linalg.norm(X[:512], axis=1).mean()))

# ---- UMAP (dimensionality reduction) ----
umap_model = umap.UMAP(
    n_neighbors=12,
    n_components=12,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True,
    verbose=True,
)

mcs = max(100, int(round(0.002 * n_docs)))
# ---- HDBSCAN (density clustering) ----
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=mcs,             # sweet spot for Reddit: 120
    min_samples=10,                 # ties to min_cluster_size (robust)
    metric="euclidean",               # operate in UMAP space
    cluster_selection_method="leaf",
    approx_min_span_tree=True,
    prediction_data=True
)

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.5,
    max_features=40_000,
    dtype=np.float32,
    strip_accents="unicode",
    token_pattern=r"(?u)\b(?:[ru]/[A-Za-z0-9_]+|[#@]\w+|\w[\w'-]{2,})\b"
)

st = SentenceTransformer("all-mpnet-base-v2")
np.save("round_11/embeddings_fp32_l2.npy", X)

umap_2d = umap.UMAP(
    n_neighbors=10, n_components=2, min_dist=0.0,
    metric="cosine", random_state=42
)
coords_2d = umap_2d.fit_transform(X)
np.save(f"{current_round_path}/umap2d.npy", coords_2d.astype("float32"))

# ---- 3. Fit BERTopic ----
topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics=None,
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)
topics_unique, probs_unique = topic_model.fit_transform(train_docs, embeddings=X)

np.save("round_11/train_topics_unique.npy", topics_unique)
topics_unique = np.asarray(topics_unique, dtype=np.int32)   # ensure ndarray
map_o2u = np.asarray(map_o2u, dtype=np.int64)         # ensure integer array

assert map_o2u.max() < topics_unique.shape[0]

# expand back to originals
topics_original = topics_unique[map_o2u]
np.save("round_11/train_topics_original.npy", topics_original)


# ---- 4. Save the topics ----
df = topic_model.get_topic_info()
df.to_csv("round_11/topics.csv", index=False)

# Save the probabilities
np.save("round_11/doc_topic_proba_f16.npy", probs_unique.astype("float16"))
probs_kept = probs_unique[map_o2u]
np.save("round_11/doc_topic_proba_kept_f16.npy", probs_kept.astype("float16"))
assert probs_unique.shape[0] == len(train_docs)
assert probs_kept.shape[0]   == kept_idx.shape[0]

# Save the model
# 1) Save BERTopic without the embedder and probs, but with c-TF-IDF
topic_model.probabilities_ = None
topic_model.save("round_11/bertopic_no_embed", save_ctfidf=True, save_embedding_model=False)

# 2) Save the SentenceTransformer separately in safetensors format
st.save_pretrained("round_11/embedder", safe_serialization=True)

labels = topic_model.hdbscan_model.labels_
n_clusters = (labels >= 0).sum() and len(set(labels[labels >= 0]))
noise_ratio = np.mean(labels == -1)
print("Clusters:", n_clusters, "Noise ratio:", round(noise_ratio, 3))

raw = topic_model.hdbscan_model.labels_
print("HDBSCAN clusters (pre-reduction):", len(set(raw[raw >= 0])))

df = topic_model.get_topic_info()
print("Final topics (post-reduction):", (df.Topic >= 0).sum())

print("n_docs:", len(train_docs))

# 1) How many docs survived detection? (log this inside preprocess too)
# Already printed, but do again:
survivor_ratio = len(train_docs)

# 2) Duplicates?
uniq = list(dict.fromkeys(train_docs))
print("dup_ratio:", round(1 - len(uniq)/len(train_docs), 3))
all_processed_docs = uniq

# 3) Length distribution (short corpora often collapse)
lens = np.array([len(s.split()) for s in all_processed_docs])
print("median_len:", int(np.median(lens)), "| p10:", int(np.percentile(lens,10)), "| p90:", int(np.percentile(lens,90)))

# 4) Did cleaning kill supplement tokens?
probe = re.compile(r"\b(5-htp|bpc-?157|b12|nmn|nad\+|nr\b|c60\b|mk-?677|akg\b|rapamycin|metformin|semaglutide)\b", re.I)
hits = sum(bool(probe.search(s)) for s in all_processed_docs[:100_000])
print("probe_hits_in_first_100k:", hits)

# 5) Sanity on min_cluster_size actually used
n_docs = len(all_processed_docs)
print("computed_min_cluster_size:", mcs)


json.dump({
  "n_docs_original": int(pd.read_json(
      raw_path,
      lines=True).shape[0]),
  "n_docs_kept": int(np.load(f"round_4/kept_indices.npy").size),
  "n_docs_unique": int(len(train_docs)),
  "min_cluster_size": int(mcs),
  "umap": {"n_neighbors":12,"n_components":12,"min_dist":0.0,"metric":"cosine","random_state":42},
    "hdbscan": {
        "min_cluster_size": mcs,
        "min_samples": 10,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "approx_min_span_tree": True,
        "prediction_data": True
    },
  "embedding_model": "all-mpnet-base-v2 (normalized)",
  "vectorizer": {"ngram_range":[1,2],"min_df":3,"max_df":0.5,"max_features":40000},
    "git_commit": "",
}, open(f"{current_round_path}/manifest.json","w"), indent=2)


# Load raw in original order
df_raw = pd.read_json(raw_path, lines=True).reset_index(drop=True)
sentinel = "||".join(df_raw["title"].astype(str).head(50).tolist() +
                     df_raw["title"].astype(str).tail(50).tolist())
print("raw_hash:", hashlib.sha1(sentinel.encode()).hexdigest())

# Base kept-frame (K rows, aligned 1:1 with topics_K/ts_K)
need_cols = ["subreddit","score","num_comments","author","link_flair_text","selftext",
             "title","permalink","url","is_self","id"]
for c in need_cols:
    if c not in df_raw.columns: df_raw[c] = np.nan

topics_K = np.load(f"{current_round_path}/train_topics_original.npy") # (K,)

meta = df_raw.iloc[kept_idx][need_cols].reset_index(drop=True)
meta.insert(0, "kept_index", np.arange(len(kept_idx), dtype=np.int32))
meta["orig_index"]    = kept_idx.astype(np.int32)
meta["unique_index"]  = map_o2u.astype(np.int32)
meta["topic_hard"]    = topics_K.astype(np.int32)
meta["ts_seconds"]    = ts_K.astype(np.int64)
meta["ts_utc"]        = pd.to_datetime(meta["ts_seconds"], unit="s", utc=True, errors="coerce")

# Dup group size (how many kept rows share this unique text)
dup_sizes = np.bincount(map_o2u, minlength=len(train_docs)).astype(np.int32)
meta["dup_group_size"] = dup_sizes[meta["unique_index"]].astype(np.int32)

# Add soft signals with saved probs
probs_path = f"{current_round_path}/doc_topic_proba_kept_f16.npy" # Fix this when calculating probs
if os.path.exists(probs_path):
    probs = np.load(probs_path).astype("float32")   # (K, T)
    meta["top_prob"] = probs.max(axis=1).astype("float32")
    # entropy in nats
    p = probs.clip(1e-12, 1.0)
    meta["entropy"] = (- (p * np.log(p)).sum(axis=1)).astype("float32")

# Light dtypes to cut size
meta["subreddit"] = meta["subreddit"].astype("category")
for c in ("author","link_flair_text"):
    meta[c] = meta[c].astype("category")

assert meta.shape[0] == kept_idx.shape[0] == map_o2u.shape[0] == ts_K.shape[0] == topics_K.shape[0], \
    (meta.shape, kept_idx.shape, map_o2u.shape, ts_K.shape, topics_K.shape)

# Save compactly
meta.to_parquet(f"{current_round_path}/kept_metadata.parquet", index=False)